# 03. Dense → Hybrid(BM25 + RRF) → Rerank (실습 2 · Step 7~12)

**이 노트북의 목표:** 02의 Dense(의미) 검색이 **고유명사·약어 질문**에서 무엇을 놓치는지 확인하고, 이를 **BM25(키워드) 검색 → RRF로 합치기 → Reranker로 재정렬** 순서로 보완한다. 단계마다 성능을 **같은 질문 세트, 같은 지표**로 재서 비교한다.

```
질문 ─┬─ Dense (BGE-M3 + Qdrant) ─ top-50 ─┐
      │                                      ├─ RRF 합치기 ─ top-20 ─ BGE-Reranker ─ 최종 top-5
      └─ BM25 (키워드, bm25s)     ─ top-50 ─┘
```

| 단계 | 잘하는 것 | 약한 것 |
|---|---|---|
| **Dense** | 표현이 달라도 뜻이 같으면 찾음 ("고대역폭 메모리" ↔ "HBM") | 처음 보는 고유명사·약어 (MLPerf, CoWoS) |
| **BM25** | 질문에 나온 단어가 **그대로** 있는 문서 | 뜻은 같지만 단어가 다르면 못 찾음, 한국어 조사 처리 |
| **RRF** | 두 결과의 **순위**만 보고 합침 → 점수 단위가 달라도 OK | 둘 다 놓친 건 못 살림 |
| **Reranker** | 질문과 청크를 **한 쌍으로 같이 읽고** 관련도 채점 → 가장 정확 | 느림 → 후보 몇십 개에만 사용 |

> 사전 준비: 02 노트북을 실행해서 `outputs/embeddings.npy`와 Qdrant `policy_reports` 컬렉션이 있어야 한다.

## Step 7. 준비: 청크 · 모델 · Dense 검색 불러오기

In [1]:
# ─────────────────────────────────────────────────────────────
# Step 7-1. 라이브러리 불러오기
#
# ⚠️ import 순서가 중요하다 (macOS)
#   torch(FlagEmbedding)를 faiss·bm25s보다 "먼저" 불러와야 한다.
#   faiss를 먼저 불러오면 "OMP: Error #15 ... libomp.dylib already initialized" 로 커널이 바로 죽는다.
#   (02에서 본 OpenMP 충돌과 같은 원인. 이 노트북은 faiss를 쓰지 않지만 습관처럼 지켜 둔다)
# ─────────────────────────────────────────────────────────────
import logging

logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)  # macOS 경고 숨김

import torch
from FlagEmbedding import BGEM3FlagModel, FlagReranker   # 임베딩 모델 + 재정렬 모델

import json
import re
import time
from pathlib import Path

import bm25s            # 빠른 BM25 키워드 검색 라이브러리
import numpy as np
import pandas as pd
from qdrant_client import QdrantClient, models

PROJECT_ROOT = Path("..").resolve()
OUT_DIR = PROJECT_ROOT / "outputs"

with open(OUT_DIR / "chunks.jsonl", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f]
texts = [c["text"] for c in chunks]

assert all(c["chunk_id"] == i for i, c in enumerate(chunks)), "chunk_id 순서가 어긋남! 01 노트북을 다시 실행하세요."
print(f"✅ 청크 로딩 완료: {len(chunks)}개")

✅ 청크 로딩 완료: 2197개


In [2]:
# ─────────────────────────────────────────────────────────────
# Step 7-2. BGE-M3 모델 + Qdrant 연결
#   - 질문을 벡터로 바꾸려면 02와 "같은 모델"이 필요하다 (청크 벡터는 Qdrant에 이미 들어 있음)
#   - Qdrant 컬렉션의 포인트 수가 청크 수와 다르면 02를 다시 실행해야 한다
# ─────────────────────────────────────────────────────────────
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

embed_model = BGEM3FlagModel("BAAI/bge-m3", devices=DEVICE, use_fp16=(DEVICE != "cpu"), normalize_embeddings=True)

client = QdrantClient(host="localhost", port=6333, timeout=60, check_compatibility=False)
COLLECTION = "policy_reports"
n_points = client.count(COLLECTION, exact=True).count
assert n_points == len(chunks), f"Qdrant {n_points}개 ≠ 청크 {len(chunks)}개 → 02 노트북을 다시 실행하세요."

print(f"✅ BGE-M3 준비 완료 (device={DEVICE}) / Qdrant '{COLLECTION}' {n_points}개 확인")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ BGE-M3 준비 완료 (device=mps) / Qdrant 'policy_reports' 2197개 확인


In [3]:
# ─────────────────────────────────────────────────────────────
# Step 7-3. Dense 검색 함수 (02의 qdrant_search와 같음)
#
# 이번 노트북의 모든 검색 함수는 "chunk_id 목록(1등부터 순서대로)"을 돌려주도록 통일한다.
#   → 나중에 RRF로 합치거나 성능을 잴 때 같은 방식으로 다룰 수 있다.
# ─────────────────────────────────────────────────────────────
def embed_query(text: str) -> list[float]:
    """질문 1개 → 1024차원 벡터 (Qdrant에 넘기기 위해 리스트로)"""
    return embed_model.encode([text], max_length=512)["dense_vecs"][0].tolist()


def dense_search(query: str, k: int = 50, query_filter: models.Filter | None = None) -> list[int]:
    """의미가 가까운 청크 k개의 chunk_id 목록 (1등부터)"""
    response = client.query_points(
        collection_name=COLLECTION, query=embed_query(query), query_filter=query_filter, limit=k, with_payload=False,
    )
    return [int(p.id) for p in response.points]


print("✅ Dense 검색 함수 준비 완료: dense_search()")
print("   테스트 'HBM이란?' top-3:", dense_search("HBM이란?", k=3))

✅ Dense 검색 함수 준비 완료: dense_search()


   테스트 'HBM이란?' top-3: [2168, 1754, 2165]


## Step 8. BM25 키워드 검색

**BM25**는 "질문에 나온 단어가 이 청크에 몇 번 나오나 + 그 단어가 얼마나 희귀한가"로 점수를 매긴다.
- "MLPerf"처럼 **드물게 나오는 단어**가 들어 있으면 점수가 크게 오른다 (희귀할수록 중요)
- "AI", "기술"처럼 **어디에나 있는 단어**는 점수에 거의 영향이 없다

BM25는 글을 **단어(토큰) 단위로 잘라서** 비교한다. 그래서 **어떻게 자르느냐(토크나이저)**가 성능을 크게 좌우한다. 한국어에는 영어에 없는 문제가 두 가지 있다.

| 문제 | 예 | 띄어쓰기로만 자르면 |
|---|---|---|
| ① 조사가 붙는다 | "HBM이란", "HBM은", "HBM의" | 셋 다 서로 다른 단어 → "HBM"과 매칭 안 됨 |
| ② 띄어쓰기가 사라진 문서 (01에서 본 삼일PwC) | "데이터센터용칩시장에서AI" | 통째로 한 단어 → 아무것도 매칭 안 됨 |

그래서 토크나이저 두 개를 만들어 비교한다.
- **공백 토크나이저:** 띄어쓰기로만 자른다 (기준선)
- **한국어 혼합 토크나이저:** ① 한글과 영문·숫자를 분리하고 ② 한글은 **2글자씩 겹쳐 자른다(bigram)**. 형태소 분석기 없이 조사·띄어쓰기 문제를 동시에 줄이는 간단한 방법이다.

In [4]:
# ─────────────────────────────────────────────────────────────
# Step 8-1. 토크나이저 2개 만들기
# ─────────────────────────────────────────────────────────────
def tokenize_whitespace(text: str) -> list[str]:
    """기준선: 소문자로 바꾸고 띄어쓰기로만 자른다."""
    return text.lower().split()


def tokenize_korean(text: str) -> list[str]:
    """
    한국어 혼합 토크나이저
      1) [가-힣]+ (한글 덩어리)와 [a-z0-9]+ (영문·숫자 덩어리)를 따로 뽑는다
         → "HBM이란" 이 "hbm" + "이란" 으로 나뉜다 (조사 분리 효과)
      2) 한글 덩어리는 2글자씩 겹쳐 자른다 (bigram)
         → "반도체시장" → "반도", "도체", "체시", "시장"
         → 띄어쓰기가 없어도 "시장", "반도" 같은 조각이 매칭된다
    """
    tokens = []
    for word in re.findall(r"[가-힣]+|[a-z0-9]+", text.lower()):
        if "가" <= word[0] <= "힣" and len(word) >= 2:
            tokens += [word[i:i + 2] for i in range(len(word) - 1)]   # 한글 bigram
        else:
            tokens.append(word)                                        # 영문·숫자·한 글자 한글은 그대로
    return tokens


for sample in ["HBM이란 무엇인가?", "데이터센터용칩시장에서AI 가속기가"]:
    print(f"📝 원문: {sample}")
    print(f"   공백     → {tokenize_whitespace(sample)}")
    print(f"   한국어혼합 → {tokenize_korean(sample)}\n")

print("✅ 토크나이저 준비 완료: tokenize_whitespace(), tokenize_korean()")

📝 원문: HBM이란 무엇인가?
   공백     → ['hbm이란', '무엇인가?']
   한국어혼합 → ['hbm', '이란', '무엇', '엇인', '인가']

📝 원문: 데이터센터용칩시장에서AI 가속기가
   공백     → ['데이터센터용칩시장에서ai', '가속기가']
   한국어혼합 → ['데이', '이터', '터센', '센터', '터용', '용칩', '칩시', '시장', '장에', '에서', 'ai', '가속', '속기', '기가']

✅ 토크나이저 준비 완료: tokenize_whitespace(), tokenize_korean()


In [5]:
# ─────────────────────────────────────────────────────────────
# Step 8-2. BM25 인덱스 2개 만들기 (토크나이저별로 하나씩)
#
# bm25s 사용법:
#   retriever = bm25s.BM25()
#   retriever.index([청크0의 토큰목록, 청크1의 토큰목록, ...])   ← 넣는 순서 = 내부 번호 = chunk_id
#   ids, scores = retriever.retrieve([질문 토큰목록], k=10)
# 청크 2천 개라 1초도 안 걸려서 파일로 저장(캐시)하지 않는다.
# ─────────────────────────────────────────────────────────────
TOKENIZERS = {"공백": tokenize_whitespace, "한국어혼합": tokenize_korean}
bm25_indexes = {}

for name, tokenize in TOKENIZERS.items():
    start = time.time()
    retriever = bm25s.BM25()   # 기본값 k1=1.5, b=0.75 (BM25 표준 파라미터)
    retriever.index([tokenize(t) for t in texts], show_progress=False)
    bm25_indexes[name] = retriever
    vocab = len(retriever.vocab_dict)
    print(f"✅ BM25[{name}] 인덱스 준비 완료: 단어 종류 {vocab:,}개, {time.time() - start:.2f}초")

✅ BM25[공백] 인덱스 준비 완료: 단어 종류 48,346개, 0.07초


✅ BM25[한국어혼합] 인덱스 준비 완료: 단어 종류 24,604개, 0.16초


In [6]:
# ─────────────────────────────────────────────────────────────
# Step 8-3. BM25 검색 함수
#
# weight_mask: 청크마다 1(검색 허용) 또는 0(제외)을 담은 배열
#   → Qdrant 필터처럼 "반도체 문서만" 검색할 때 쓴다 (Step 12에서 사용)
# 점수가 0인 결과 = 질문 단어가 하나도 안 겹친 청크 → 의미 없으니 버린다
# ─────────────────────────────────────────────────────────────
def bm25_search(query: str, k: int = 50, tokenizer: str = "한국어혼합", mask: np.ndarray | None = None) -> list[int]:
    """질문 단어가 많이·희귀하게 겹치는 청크 k개의 chunk_id 목록 (1등부터)"""
    retriever = bm25_indexes[tokenizer]
    query_tokens = TOKENIZERS[tokenizer](query)
    ids, scores = retriever.retrieve([query_tokens], k=k, show_progress=False, weight_mask=mask)
    return [int(i) for i, s in zip(ids[0], scores[0]) if s > 0]


print("✅ BM25 검색 함수 준비 완료: bm25_search()")

✅ BM25 검색 함수 준비 완료: bm25_search()


In [7]:
# ─────────────────────────────────────────────────────────────
# Step 8-4. 결과를 표로 보는 함수 + 토크나이저 비교
#   "HBM이란 무엇인가?" → 공백 토크나이저는 'hbm이란', '무엇인가?' 라는 "단어"를 찾는데,
#   이런 단어는 어떤 청크에도 그대로 들어 있지 않아서 결과가 0개가 된다.
# ─────────────────────────────────────────────────────────────
def show(ids: list[int], keyword: str | None = None, k: int = 5, preview: int = 45) -> pd.DataFrame:
    """chunk_id 목록 → 표. keyword를 주면 그 단어가 청크에 들어 있는지(✅/❌) 표시"""
    if not ids:   # 검색 결과가 0개면 (예: BM25에서 질문 단어가 하나도 안 겹칠 때) 빈 표 대신 안내
        return pd.DataFrame({"결과": ["(검색 결과 0개: 질문 단어와 겹치는 청크가 하나도 없음)"]})
    rows = []
    for rank, cid in enumerate(ids[:k], start=1):
        c = chunks[cid]
        row = {"순위": rank, "발행처": c["publisher"], "연도": c["year"], "쪽": c["page"], "chunk_id": cid}
        if keyword:
            row[f"'{keyword}' 포함"] = "✅" if keyword.lower() in c["text"].lower() else "❌"
        row["내용 미리보기"] = c["text"][:preview].replace("\n", " ")
        rows.append(row)
    return pd.DataFrame(rows).set_index("순위")


query, kw = "HBM이란 무엇인가?", "HBM"
for name in TOKENIZERS:
    print(f"\n🔎 BM25[{name}] — {query}")
    print(show(bm25_search(query, tokenizer=name), keyword=kw).to_string())


🔎 BM25[공백] — HBM이란 무엇인가?
                                  결과
0  (검색 결과 0개: 질문 단어와 겹치는 청크가 하나도 없음)

🔎 BM25[한국어혼합] — HBM이란 무엇인가?
         발행처    연도    쪽  chunk_id 'HBM' 포함                                        내용 미리보기
순위                                                                                       
1   반도체특별위원회  2025   87      2058        ❌  제4장 K-AI반도체 산업생태계 강국도약 가이드라인 성능이 나타난다. 즉, AI 
2       ETRI  2020  125      1293        ❌  있음 l 학계를 중심으로 멀티 에이전트 및 엣지 노드간 분산 추론 방법 등의 연구
3   반도체특별위원회  2025   27      1892        ✅  (Fabless) NVIDIA, AMD, Google, AWS 등 세계 주도 Hu
4       ETRI  2020  123      1286        ❌  ④ AI 기반 영상 콘텐츠 지능분석 및 자동 생성 기술 (1) AI 서비스를 위한
5       ETRI  2020  229      1549        ❌  - (네트워크 기반 침입탐지) 인가되지 않은 사용자의 불법 접근이나, 정보자원을 


## Step 9. Dense가 놓치는 질문 찾기 (평가 세트 만들기)

"어느 방식이 더 좋다"를 말하려면 **같은 질문 세트**로 **숫자**를 재야 한다.

정답 라벨을 사람이 다는 대신, 이번에는 **간단한 근사 평가**를 쓴다.
- 질문마다 **핵심 키워드**를 하나 정한다 (예: "MLPerf 벤치마크 결과" → `MLPerf`)
- 검색 결과 청크에 그 키워드가 **들어 있으면 관련 있음**으로 본다

| 지표 | 계산 | 의미 |
|---|---|---|
| **P@5** (Precision@5) | top-5 중 키워드가 든 청크 비율 | 상위 결과가 얼마나 "깨끗한가" |
| **MRR@10** | 키워드가 든 첫 청크가 k등이면 1/k (10등 안에 없으면 0) | 첫 정답이 얼마나 위에 있나 |
| **최저 P@5** | 질문들 중 가장 나쁜 P@5 | "크게 망하는 질문"이 있는가 |

> ⚠️ **이 지표의 한계:** "키워드가 들어 있나"로 채점하므로 **키워드 매칭 방식인 BM25에 유리**하다. 또 키워드가 있어도 질문에 대한 답이 아닐 수 있다. 진짜 답변 품질은 04에서 RAGAS로 평가한다. 여기서는 "단계별 경향"을 보는 용도다.

In [8]:
# ─────────────────────────────────────────────────────────────
# Step 9-1. 평가 질문 세트 (핵심 키워드, 질문)
#
# 고유명사·약어 위주로 고르되, Dense가 잘하는 일반 개념 질문도 섞었다.
# 키워드가 데이터에 몇 번 나오는지(정답 후보 수)도 같이 본다.
#   ※ 데이터에 아예 없는 키워드(예: GB200)는 평가가 안 되므로 뺐다.
# ─────────────────────────────────────────────────────────────
EVAL_SET = [
    ("CXL", "CXL 메모리 기술이란?"),
    ("PIM", "PIM 반도체의 장점"),
    ("TSMC", "TSMC 파운드리 점유율"),
    ("리벨리온", "리벨리온 NPU 제품"),
    ("퓨리오사", "퓨리오사AI 칩 개발"),
    ("엑사원", "엑사원 모델 공개"),
    ("MLPerf", "MLPerf 벤치마크 결과"),
    ("CoWoS", "CoWoS 패키징 공급 부족"),
    ("딥시크", "딥시크 R1 출시 영향"),
    ("블랙웰", "엔비디아 블랙웰 출시"),
    ("TPU", "구글 TPU 개발 현황"),
    ("소버린", "소버린 AI 전략"),
    ("온디바이스", "온디바이스 AI 칩"),
    ("HBM", "HBM이란 무엇인가?"),
    ("NPU", "NPU와 GPU의 차이"),
]

pd.DataFrame(
    [{"키워드": kw, "질문": q, "키워드 든 청크 수": sum(kw.lower() in t.lower() for t in texts)} for kw, q in EVAL_SET]
).set_index("키워드").T

키워드,CXL,PIM,TSMC,리벨리온,퓨리오사,엑사원,MLPerf,CoWoS,딥시크,블랙웰,TPU,소버린,온디바이스,HBM,NPU
질문,CXL 메모리 기술이란?,PIM 반도체의 장점,TSMC 파운드리 점유율,리벨리온 NPU 제품,퓨리오사AI 칩 개발,엑사원 모델 공개,MLPerf 벤치마크 결과,CoWoS 패키징 공급 부족,딥시크 R1 출시 영향,엔비디아 블랙웰 출시,구글 TPU 개발 현황,소버린 AI 전략,온디바이스 AI 칩,HBM이란 무엇인가?,NPU와 GPU의 차이
키워드 든 청크 수,19,15,34,7,4,6,3,5,28,4,19,13,34,69,62


In [9]:
# ─────────────────────────────────────────────────────────────
# Step 9-2. 채점 함수
# ─────────────────────────────────────────────────────────────
def score_one(keyword: str, ids: list[int]) -> dict:
    """검색 결과(chunk_id 목록) 하나를 P@5, MRR@10으로 채점"""
    hits = [keyword.lower() in texts[i].lower() for i in ids[:10]]   # 순위별로 키워드 포함 여부 True/False
    p_at_5 = sum(hits[:5]) / 5
    mrr = next((1 / rank for rank, hit in enumerate(hits, start=1) if hit), 0.0)  # 첫 True의 순위 → 1/순위
    return {"P@5": p_at_5, "MRR@10": mrr}


def evaluate(search_fn, name: str) -> pd.DataFrame:
    """EVAL_SET 전체를 search_fn으로 검색해서 질문별 점수표를 만든다. 질문당 평균 검색 시간도 잰다."""
    rows = []
    for kw, q in EVAL_SET:
        start = time.perf_counter()
        ids = search_fn(q)
        ms = (time.perf_counter() - start) * 1000
        rows.append({"방식": name, "키워드": kw, **score_one(kw, ids), "ms": ms})
    return pd.DataFrame(rows)


def summarize(df: pd.DataFrame) -> pd.DataFrame:
    """질문별 점수표 → 방식별 요약 (평균 P@5, 평균 MRR, 최저 P@5, 평균 시간)"""
    g = df.groupby("방식", sort=False)
    return pd.DataFrame({
        "P@5": g["P@5"].mean(),
        "MRR@10": g["MRR@10"].mean(),
        "최저 P@5": g["P@5"].min(),
        "질문당 ms": g["ms"].mean(),
    }).round(3)


print("✅ 채점 함수 준비 완료: score_one(), evaluate(), summarize()")

✅ 채점 함수 준비 완료: score_one(), evaluate(), summarize()


In [10]:
# ─────────────────────────────────────────────────────────────
# Step 9-3. Dense vs BM25(공백) vs BM25(한국어혼합)
#
# 각 방식이 top-5 중 몇 개나 키워드 청크를 찾았는지 질문별로 비교한다.
# (Dense는 질문 임베딩 시간이 포함돼서 BM25보다 느리게 나온다)
# ─────────────────────────────────────────────────────────────
results = pd.concat([
    evaluate(lambda q: dense_search(q, k=10), "Dense"),
    evaluate(lambda q: bm25_search(q, k=10, tokenizer="공백"), "BM25(공백)"),
    evaluate(lambda q: bm25_search(q, k=10, tokenizer="한국어혼합"), "BM25(한국어혼합)"),
])

per_query = results.pivot(index="키워드", columns="방식", values="P@5")[["Dense", "BM25(공백)", "BM25(한국어혼합)"]]
per_query = per_query.loc[[kw for kw, _ in EVAL_SET]]   # 표 순서를 EVAL_SET 순서로

print("📊 질문별 P@5 (top-5 중 키워드 청크 비율)")
print(per_query.to_string())
print("\n📊 방식별 요약")
summarize(results)

📊 질문별 P@5 (top-5 중 키워드 청크 비율)
방식      Dense  BM25(공백)  BM25(한국어혼합)
키워드                                 
CXL       1.0       1.0          1.0
PIM       1.0       1.0          0.8
TSMC      0.8       1.0          0.8
리벨리온      0.6       0.6          1.0
퓨리오사      0.2       0.2          0.8
엑사원       0.8       0.8          1.0
MLPerf    0.0       0.6          0.2
CoWoS     0.4       0.2          0.8
딥시크       0.6       0.8          1.0
블랙웰       0.4       0.4          0.8
TPU       0.6       0.8          0.8
소버린       1.0       0.8          1.0
온디바이스     1.0       1.0          1.0
HBM       1.0       0.0          0.2
NPU       1.0       0.6          0.8

📊 방식별 요약


,P@5,MRR@10,최저 P@5,질문당 ms
방식,,,,
Dense,0.693,0.944,0.0,34.523
BM25(공백),0.653,0.933,0.0,0.034
BM25(한국어혼합),0.800,0.956,0.2,0.034


In [11]:
# ─────────────────────────────────────────────────────────────
# Step 9-4. (눈으로 확인) Dense가 크게 놓친 질문: "MLPerf 벤치마크 결과"
#   MLPerf = AI 칩 성능을 재는 업계 표준 벤치마크 이름.
#   Dense는 "벤치마크 결과"라는 뜻에 끌려 다른 벤치마크 청크를 가져오고,
#   BM25는 'mlperf'라는 희귀 단어가 그대로 든 청크를 찾는다.
# ─────────────────────────────────────────────────────────────
query, kw = "MLPerf 벤치마크 결과", "MLPerf"
print(f"🔎 Dense — {query}")
print(show(dense_search(query), keyword=kw).to_string())
print(f"\n🔎 BM25(한국어혼합) — {query}")
print(show(bm25_search(query), keyword=kw).to_string())

🔎 Dense — MLPerf 벤치마크 결과
     발행처    연도   쪽  chunk_id 'MLPerf' 포함                                        내용 미리보기
순위                                                                                     
1   SPRi  2025  19        52           ❌  세일즈포스, 실제 사용 환경에서 LLM 성능 검증을 위한 벤치마크 개발 n 세일즈
2   SPRi  2025  19        53           ❌  사용 환경에서 LLM의 복잡한 추론이나 장기적 맥락 이해, 도구 활용 능력의 평가
3   SPRi  2025  17       228           ❌  ML커먼스, AI 안전성 평가 벤치마크 ‘AI루미네이트’ 발표 n ML커먼스가 물
4   SPRi  2026  23       328           ❌  ∙일례로 성능이 뛰어난 최신 모델이 공통으로 오답을 내는 문항이 있다면 문항 자체
5   SPRi  2025  21       152           ❌  ** 체화 추론 벤치마크 평균 점수: 제미나이 로보틱스-ER 1.5(62.8), 

🔎 BM25(한국어혼합) — MLPerf 벤치마크 결과
           발행처    연도   쪽  chunk_id 'MLPerf' 포함                                        내용 미리보기
순위                                                                                           
1   KDB미래전략연구소  2024   7      2154           ✅  2024. 7 제824호  9 ○엔비디아의 AI GPU는 MLPerf4) 벤치마크
2         SPRi  2025  21       489           

### 📝 Step 8~9 관찰 메모
- **토크나이저가 BM25 성능을 좌우한다.** 공백 토크나이저는 "HBM이란 무엇인가?"에서 **결과가 0개**였다(`hbm이란`, `무엇인가?`라는 단어가 어디에도 없음). 한국어혼합 토크나이저로 바꾸자 평균 P@5가 **0.653 → 0.800**으로 올랐다.
- 한글 bigram 덕분에 띄어쓰기가 사라진 삼일PwC 문장도 `시장`, `가속` 같은 조각으로 매칭된다. 단어 종류도 48,346개에서 24,604개로 줄었다(같은 단어의 조사 붙은 변형들이 하나로 모임).
- **Dense가 크게 놓친 질문:** `MLPerf`(0.0), `퓨리오사`(0.2), `CoWoS`(0.4), `블랙웰`(0.4). "MLPerf 벤치마크 결과"에서 Dense top-5는 전부 **다른 벤치마크** 기사였다. "벤치마크"라는 뜻에 끌려가고 `MLPerf`라는 고유명사는 무시한 것이다. → README에서 말한 "고유명사 질문에서 Dense가 놓치는 부분"을 수치로 확인했다.
- **반대로 BM25가 놓친 질문:** `HBM이란 무엇인가?`(한국어혼합 0.2). `무엇`, `인가` 같은 흔한 bigram이 점수를 흩뜨렸다. Dense는 1.0이다. → **두 방식이 서로 다른 질문에서 실패하므로 합칠 가치가 있다.**
- `MLPerf`는 공백 토크나이저(0.6)가 한국어혼합(0.2)보다 나았다. "벤치마크 결과"가 bigram 4개로 쪼개지면서 `mlperf` 한 단어의 비중이 상대적으로 작아졌기 때문이다. 만능 토크나이저는 없다.
- 이 지표는 키워드 매칭 방식이라 BM25에 유리하다는 점을 감안해야 한다.

## Step 10. Hybrid: RRF로 Dense + BM25 합치기

Dense 점수(코사인, 0.3~0.7)와 BM25 점수(0~30 이상)는 **단위가 달라서** 그냥 더하면 BM25가 다 이겨버린다.

**RRF(Reciprocal Rank Fusion)**는 점수는 버리고 **순위만** 쓴다.

$$\text{RRF}(d) = \sum_{\text{검색 방식}} \frac{1}{k + \text{순위}(d)}$$

- 두 방식 모두에서 상위권 → 점수 2번 받음 → 최종 1등 후보
- 한쪽에서만 1등 → 1/(60+1) ≈ 0.0164 한 번만
- `k=60`은 RRF 논문의 기본값. 1등과 2등의 차이를 너무 크게 만들지 않는 "완충 장치"다.

In [12]:
# ─────────────────────────────────────────────────────────────
# Step 10-1. RRF 함수 + 작은 예시로 계산 확인
# ─────────────────────────────────────────────────────────────
RRF_K = 60


def rrf_fuse(ranked_lists: list[list[int]], k: int = RRF_K) -> list[int]:
    """여러 검색 결과(chunk_id 순위 목록)를 RRF 점수로 합쳐서 새 순위 목록을 돌려준다."""
    scores = {}
    for ranked in ranked_lists:
        for rank, cid in enumerate(ranked, start=1):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank)   # 등장할 때마다 점수 누적
    return sorted(scores, key=scores.get, reverse=True)             # 점수 높은 순


# 예시: Dense 결과 [A, B, C], BM25 결과 [C, D, A]
#   A: 1/(60+1) + 1/(60+3) = 0.0323   ← 양쪽에 다 있음
#   C: 1/(60+3) + 1/(60+1) = 0.0323   ← 양쪽에 다 있음
#   B: 1/(60+2)            = 0.0161
#   D: 1/(60+2)            = 0.0161
print("예시 RRF 결과:", rrf_fuse([["A", "B", "C"], ["C", "D", "A"]]), " ← 양쪽에 다 나온 A, C가 위로")
print("✅ RRF 함수 준비 완료: rrf_fuse()")

예시 RRF 결과: ['A', 'C', 'B', 'D']  ← 양쪽에 다 나온 A, C가 위로
✅ RRF 함수 준비 완료: rrf_fuse()


In [13]:
# ─────────────────────────────────────────────────────────────
# Step 10-2. Hybrid 검색 함수
#   각 방식에서 넉넉하게 50개씩 후보를 뽑아서 합친다.
#   (5개씩만 뽑으면 한쪽 6등에 숨어 있던 좋은 청크가 아예 기회를 못 얻는다)
# ─────────────────────────────────────────────────────────────
CANDIDATES = 50


def hybrid_search(query: str, k: int = 50, query_filter: models.Filter | None = None, mask: np.ndarray | None = None) -> list[int]:
    dense_ids = dense_search(query, k=CANDIDATES, query_filter=query_filter)
    bm25_ids = bm25_search(query, k=CANDIDATES, mask=mask)
    return rrf_fuse([dense_ids, bm25_ids])[:k]


print("✅ Hybrid 검색 함수 준비 완료: hybrid_search()")

✅ Hybrid 검색 함수 준비 완료: hybrid_search()


In [14]:
# ─────────────────────────────────────────────────────────────
# Step 10-3. Hybrid 성능 측정 → 앞 단계와 비교
# ─────────────────────────────────────────────────────────────
results = pd.concat([results, evaluate(lambda q: hybrid_search(q, k=10), "Hybrid(RRF)")])
summarize(results)

,P@5,MRR@10,최저 P@5,질문당 ms
방식,,,,
Dense,0.693,0.944,0.0,34.523
BM25(공백),0.653,0.933,0.0,0.034
BM25(한국어혼합),0.800,0.956,0.2,0.034
Hybrid(RRF),0.787,0.967,0.2,32.808


## Step 11. Reranker로 재정렬

Dense는 질문과 청크를 **따로** 벡터로 만든 뒤 비교한다(Bi-encoder). 빠르지만, 질문의 어느 단어가 청크의 어느 부분과 맞는지는 보지 못한다.

**Reranker**(Cross-encoder)는 `[질문, 청크]`를 **한 번에 같이 읽고** "이 청크가 이 질문에 답이 되나?"를 0~1 점수로 매긴다. 훨씬 정확하지만, 청크마다 모델을 한 번씩 돌려야 해서 느리다.

→ 그래서 **Hybrid로 후보 20개를 빠르게 추리고 → Reranker로 20개만 정밀 채점**하는 2단계 구조를 쓴다.

모델: `BAAI/bge-reranker-v2-m3` (BGE-M3 기반 다국어 Reranker, 한국어 지원). 처음 실행하면 약 2GB를 내려받는다.

In [15]:
# ─────────────────────────────────────────────────────────────
# Step 11-1. Reranker 모델 불러오기
#   normalize=True → 점수를 0~1 사이로 바꿔준다 (sigmoid). 해석하기 쉬워진다.
# ─────────────────────────────────────────────────────────────
reranker = FlagReranker(
    "BAAI/bge-reranker-v2-m3",
    devices=DEVICE,
    use_fp16=(DEVICE != "cpu"),
    max_length=512,      # 질문 + 청크를 합쳐서 512 토큰까지 읽는다 (청크가 500자라 충분)
)
print(f"✅ Reranker 준비 완료 (device={DEVICE})")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

✅ Reranker 준비 완료 (device=mps)


In [16]:
# ─────────────────────────────────────────────────────────────
# Step 11-2. Rerank 함수 + Hybrid+Rerank 검색 함수
# ─────────────────────────────────────────────────────────────
RERANK_TOP = 20   # Reranker가 다시 채점할 후보 수 (Step 11-5 실험에서 20 = 30과 정확도가 같고 더 빨랐다)


def rerank(query: str, candidate_ids: list[int]) -> tuple[list[int], list[float]]:
    """후보 청크들을 Reranker 점수 순으로 다시 정렬. (chunk_id 목록, 점수 목록)을 돌려준다."""
    pairs = [(query, texts[cid]) for cid in candidate_ids]        # [질문, 청크] 쌍 만들기
    scores = reranker.compute_score(pairs, normalize=True)        # 쌍마다 0~1 점수
    order = np.argsort(scores)[::-1]                              # 점수 높은 순서의 위치
    return [candidate_ids[i] for i in order], [float(scores[i]) for i in order]


def hybrid_rerank_search(query: str, k: int = 10, query_filter=None, mask=None, top_n: int = RERANK_TOP) -> list[int]:
    candidates = hybrid_search(query, k=top_n, query_filter=query_filter, mask=mask)   # ① 빠르게 후보 추리기
    ids, _ = rerank(query, candidates)                                                  # ② 정밀 재채점
    return ids[:k]


print("✅ Rerank 함수 준비 완료: rerank(), hybrid_rerank_search()")

✅ Rerank 함수 준비 완료: rerank(), hybrid_rerank_search()


In [17]:
# ─────────────────────────────────────────────────────────────
# Step 11-3. (눈으로 확인) Rerank 전/후 순위 변화
#   "CoWoS 패키징 공급 부족": CoWoS = TSMC의 첨단 패키징 기술 이름.
#   Hybrid 순위에서 아래쪽에 있던 CoWoS 청크가 Reranker 후 위로 올라오는지 본다.
# ─────────────────────────────────────────────────────────────
query, kw = "CoWoS 패키징 공급 부족", "CoWoS"
candidates = hybrid_search(query, k=RERANK_TOP)
reranked, rr_scores = rerank(query, candidates)

before_rank = {cid: r for r, cid in enumerate(candidates, start=1)}
table = show(reranked, keyword=kw, k=5, preview=40)
table.insert(0, "Rerank 점수", [round(s, 3) for s in rr_scores[:5]])
table.insert(1, "Hybrid 때 순위", [before_rank[cid] for cid in reranked[:5]])

print(f"🔎 질문: {query}")
print("\n[Hybrid top-5 (Rerank 전)]")
print(show(candidates, keyword=kw, k=5, preview=40).to_string())
print("\n[Rerank 후 top-5]  ← 'Hybrid 때 순위'가 크면 아래에서 끌어올린 것")
table

🔎 질문: CoWoS 패키징 공급 부족

[Hybrid top-5 (Rerank 전)]
           발행처    연도   쪽  chunk_id 'CoWoS' 포함                                   내용 미리보기
순위                                                                                     
1   KDB미래전략연구소  2024  18      2182          ✅  시장 개화에 대응하여 자사 제품에 적합한 AI 반도체를 개발 25) 반도
2   KDB미래전략연구소  2024  18      2180          ✅  ❍ (파운드리) 글로벌 파운드리 Big3(TSMC(대만)‧삼성전자‧인텔(
3        삼일PwC  2025  70      1765          ❌  글로벌 반도체 패키징 및 테스트 시장 출처: 회사 발표, 전문가 인터뷰,
4     반도체특별위원회  2025  77      2031          ❌  상생파운드리 같은 공공 파운드리 혹은 대형 나노팹 등과 연계하여 국가 M
5        삼일PwC  2025  74      1773          ❌  공정기술이발전함에따라, Front-end 팹과장비공급업체모두더높은 기술적

[Rerank 후 top-5]  ← 'Hybrid 때 순위'가 크면 아래에서 끌어올린 것


,Rerank 점수,Hybrid 때 순위,발행처,연도,쪽,chunk_id,'CoWoS' 포함,내용 미리보기
순위,,,,,,,,
1,0.489,2,KDB미래전략연구소,2024,18,2180,✅,❍ (파운드리) 글로벌 파운드리 Big3(TSMC(대만)‧삼성전자‧인텔(
2,0.257,8,반도체특별위원회,2025,32,1910,✅,"TSMC를 중심으로 한 파운드리 역량은 세계 최고 수준이며, CoWoS와"
3,0.136,16,반도체특별위원회,2025,30,1903,✅,라이브러리･IP 생태계를 통해 설계–제조 연계를 극대화하고 있다. 설계
4,0.107,20,삼일PwC,2025,64,1755,❌,"발생할 수 있습니다. 그이유는다음과같습니다. 첫째, 수요가 시장 전망치를"
5,0.106,1,KDB미래전략연구소,2024,18,2182,✅,시장 개화에 대응하여 자사 제품에 적합한 AI 반도체를 개발 25) 반도


In [18]:
# ─────────────────────────────────────────────────────────────
# Step 11-4. Hybrid + Rerank 성능 측정
# ─────────────────────────────────────────────────────────────
results = pd.concat([results, evaluate(lambda q: hybrid_rerank_search(q, k=10), "Hybrid+Rerank")])
summarize(results)

,P@5,MRR@10,최저 P@5,질문당 ms
방식,,,,
Dense,0.693,0.944,0.0,34.523
BM25(공백),0.653,0.933,0.0,0.034
BM25(한국어혼합),0.800,0.956,0.2,0.034
Hybrid(RRF),0.787,0.967,0.2,32.808
Hybrid+Rerank,0.867,0.967,0.4,654.445


In [19]:
# ─────────────────────────────────────────────────────────────
# Step 11-5. Rerank 후보 수(top_n)를 바꾸면? → 정확도 vs 속도
#   후보가 많을수록 Reranker가 좋은 청크를 발견할 기회가 늘지만, 그만큼 느려진다.
# ─────────────────────────────────────────────────────────────
rows = []
for n in [10, 20, 30, 50]:
    df = evaluate(lambda q, n=n: hybrid_rerank_search(q, k=10, top_n=n), f"top_n={n}")
    rows.append(summarize(df))
pd.concat(rows)

,P@5,MRR@10,최저 P@5,질문당 ms
방식,,,,
top_n=10,0.840,1.000,0.2,339.014
top_n=20,0.867,0.967,0.4,642.632
top_n=30,0.867,0.967,0.4,990.460
top_n=50,0.880,0.967,0.6,1761.490


### 📝 Step 10~11 관찰 메모
- **Hybrid(RRF)** 평균 P@5는 0.787로 BM25(한국어혼합) 0.800과 비슷하다. 대신 **양쪽의 실패를 서로 메워준다.**
  - `HBM`: BM25 0.2 → Hybrid **1.0** (Dense가 살림)
  - `퓨리오사`: Dense 0.2 → Hybrid **0.6** (BM25가 살림)
  - 평균보다 "어떤 질문이 와도 크게 망하지 않는 것"이 Hybrid의 역할이다.
- **Hybrid + Rerank**가 가장 좋았다: P@5 **0.867**(Dense 대비 +0.174), 최저 P@5 **0.0 → 0.4**.
  - "CoWoS 패키징 공급 부족"에서 Hybrid 8등·16등에 묻혀 있던 CoWoS 청크를 **2등·3등**으로 끌어올렸다.
- Rerank 후 오히려 떨어진 질문도 있다(`딥시크`, `온디바이스` 1.0 → 0.8).
  - `딥시크`: 감점된 청크는 "**DeepSeek**-R1"이 영어로 적힌 청크였다. Reranker는 같은 대상을 찾았는데 키워드 채점이 한글 표기만 봐서 생긴 감점이다(**지표의 한계**).
  - `온디바이스`: 감점된 청크는 클라우드 기업의 자체 칩 개발 내용이라 실제로 덜 관련된 결과다(**Reranker의 실수**).
- **대가는 속도다.** Hybrid는 질문당 약 34ms(대부분 질문 임베딩 시간)인데, Rerank를 붙이면 질문당 **0.6초 이상**(M5 MPS 기준)이 된다.
- **후보 수(top_n) 실험:** 10 → 20에서 P@5 0.840 → 0.867로 오르고, 20 = 30은 같았다. 50은 0.880으로 조금 더 좋지만 시간이 약 2.7배 걸린다. → 기본값을 **20**으로 정했다.

## Step 12. 최종 비교 + 필터까지 합친 검색 함수

In [20]:
# ─────────────────────────────────────────────────────────────
# Step 12-1. 단계별 최종 비교표 (Before → After)
# ─────────────────────────────────────────────────────────────
final = summarize(results)
final["P@5 변화(Dense 대비)"] = (final["P@5"] - final.loc["Dense", "P@5"]).round(3)

per_query = results.pivot(index="키워드", columns="방식", values="P@5").loc[[kw for kw, _ in EVAL_SET], final.index]
print("📊 질문별 P@5")
print(per_query.to_string())
print("\n📊 방식별 요약")
final

📊 질문별 P@5
방식      Dense  BM25(공백)  BM25(한국어혼합)  Hybrid(RRF)  Hybrid+Rerank
키워드                                                             
CXL       1.0       1.0          1.0          1.0            1.0
PIM       1.0       1.0          0.8          1.0            1.0
TSMC      0.8       1.0          0.8          0.8            1.0
리벨리온      0.6       0.6          1.0          0.6            0.8
퓨리오사      0.2       0.2          0.8          0.6            0.8
엑사원       0.8       0.8          1.0          0.8            0.8
MLPerf    0.0       0.6          0.2          0.2            0.4
CoWoS     0.4       0.2          0.8          0.4            0.8
딥시크       0.6       0.8          1.0          1.0            0.8
블랙웰       0.4       0.4          0.8          0.6            0.8
TPU       0.6       0.8          0.8          0.8            1.0
소버린       1.0       0.8          1.0          1.0            1.0
온디바이스     1.0       1.0          1.0          1.0            0.8
HBM       1.0  

,P@5,MRR@10,최저 P@5,질문당 ms,P@5 변화(Dense 대비)
방식,,,,,
Dense,0.693,0.944,0.0,34.523,0.000
BM25(공백),0.653,0.933,0.0,0.034,-0.040
BM25(한국어혼합),0.800,0.956,0.2,0.034,0.107
Hybrid(RRF),0.787,0.967,0.2,32.808,0.094
Hybrid+Rerank,0.867,0.967,0.4,654.445,0.174


In [21]:
# ─────────────────────────────────────────────────────────────
# Step 12-2. 필터까지 지원하는 최종 검색 함수 search()
#
# 02의 메타데이터 필터를 Hybrid에도 적용한다.
#   - Dense 쪽: Qdrant 필터 (query_filter)
#   - BM25 쪽 : 조건에 맞는 청크만 1, 나머지 0인 mask 배열
#   → 두 쪽 모두 "같은 조건"으로 거른 뒤 RRF → Rerank
# 04(RAGAS 평가)에서도 이 함수를 그대로 가져다 쓸 예정.
# ─────────────────────────────────────────────────────────────
def search(query: str, k: int = 5, mode: str = "rerank", category: str | None = None, year_gte: int | None = None) -> list[int]:
    """
    mode: "dense" | "bm25" | "hybrid" | "rerank"
    category: "반도체" / "AI동향" / "AI전략" 중 하나 (None이면 전체)
    year_gte: 이 연도 이상만 (None이면 전체)
    """
    # ① Qdrant용 필터
    conditions = []
    if category:
        conditions.append(models.FieldCondition(key="category", match=models.MatchValue(value=category)))
    if year_gte:
        conditions.append(models.FieldCondition(key="year", range=models.Range(gte=year_gte)))
    query_filter = models.Filter(must=conditions) if conditions else None

    # ② BM25용 mask (같은 조건을 파이썬으로 계산)
    mask = None
    if conditions:
        mask = np.array([
            (category is None or c["category"] == category) and (year_gte is None or c["year"] >= year_gte)
            for c in chunks
        ], dtype=np.float32)

    if mode == "dense":
        return dense_search(query, k=k, query_filter=query_filter)
    if mode == "bm25":
        return bm25_search(query, k=k, mask=mask)
    if mode == "hybrid":
        return hybrid_search(query, k=k, query_filter=query_filter, mask=mask)
    if mode == "rerank":
        return hybrid_rerank_search(query, k=k, query_filter=query_filter, mask=mask)
    raise ValueError(f"알 수 없는 mode: {mode}")


print("✅ 최종 검색 함수 준비 완료: search(query, k, mode, category, year_gte)")

✅ 최종 검색 함수 준비 완료: search(query, k, mode, category, year_gte)


In [22]:
# ─────────────────────────────────────────────────────────────
# Step 12-3. 필터 + Hybrid + Rerank 실행 예시
#   "엔비디아 블랙웰 출시"를 ① 전체에서 ② 2026년 자료만 검색
# ─────────────────────────────────────────────────────────────
query, kw = "엔비디아 블랙웰 출시", "블랙웰"

print(f"🔎 {query}  (전체)")
print(show(search(query), keyword=kw).to_string())

print(f"\n🔎 {query}  (year >= 2026)")
filtered = search(query, year_gte=2026)
print(show(filtered, keyword=kw).to_string())

assert all(chunks[cid]["year"] >= 2026 for cid in filtered)
print("\n✅ 검증: 필터 결과 모두 year >= 2026 (Dense·BM25 양쪽 모두 필터 적용됨)")

🔎 엔비디아 블랙웰 출시  (전체)


     발행처    연도   쪽  chunk_id '블랙웰' 포함                                        내용 미리보기
순위                                                                                  
1   SPRi  2025  14       901        ✅  대륙을 AI 산업 혁명의 선두 주자로 자리매김하기 위한 AI 인프라를 구축하겠다고
2   SPRi  2026   8       793        ❌  기업 여 곳에 엔비디아의 IT H200* 구매를 승인했으나중국 정부의 구매 제한 
3   SPRi  2025  14       902        ✅  데이터센터를 구축해 공장 디지털트윈과 로봇공학 등 제조업의 다양한 활용 사례를 뒷
4   SPRi  2026  16       645        ✅  Nemotron 3 Super (Mamba)* 계층과 고급 추론 능력을 담당하는 
5   SPRi  2026  18       651        ✅  엔비디아 에서 생태계 확장 선언 , GTC 2026 엔비디아의 젠슨 황 는 에서 

🔎 엔비디아 블랙웰 출시  (year >= 2026)


     발행처    연도   쪽  chunk_id '블랙웰' 포함                                        내용 미리보기
순위                                                                                  
1   SPRi  2026   8       793        ❌  기업 여 곳에 엔비디아의 IT H200* 구매를 승인했으나중국 정부의 구매 제한 
2   SPRi  2026  16       645        ✅  Nemotron 3 Super (Mamba)* 계층과 고급 추론 능력을 담당하는 
3   SPRi  2026  18       651        ✅  엔비디아 에서 생태계 확장 선언 , GTC 2026 엔비디아의 젠슨 황 는 에서 
4   SPRi  2026  18       654        ❌  (Uber) 개 도시에 엔비디아 소프트웨어로 구동되는 자율주행 차량을 도입 예정 
5   SPRi  2026  16       644        ❌  엔비디아에이전틱 시스템을 위한 출시 ‘Nemotron 3 Super’ 엔비디아가 

✅ 검증: 필터 결과 모두 year >= 2026 (Dense·BM25 양쪽 모두 필터 적용됨)


In [23]:
# ─────────────────────────────────────────────────────────────
# Step 12-4. 결과 저장 (04에서 Before/After 비교에 사용)
#   - eval_queries.json   : 평가 질문 세트
#   - retrieval_eval.csv  : 방식·질문별 점수
# ─────────────────────────────────────────────────────────────
with open(OUT_DIR / "eval_queries.json", "w", encoding="utf-8") as f:
    json.dump([{"keyword": kw, "question": q} for kw, q in EVAL_SET], f, ensure_ascii=False, indent=2)

results.to_csv(OUT_DIR / "retrieval_eval.csv", index=False, encoding="utf-8-sig")   # utf-8-sig → 엑셀에서 한글 안 깨짐

print(f"💾 저장 완료: {OUT_DIR / 'eval_queries.json'} ({len(EVAL_SET)}개 질문)")
print(f"💾 저장 완료: {OUT_DIR / 'retrieval_eval.csv'} ({len(results)}행)")

💾 저장 완료: /Users/simhyeok/Desktop/vectordb-rag/outputs/eval_queries.json (15개 질문)
💾 저장 완료: /Users/simhyeok/Desktop/vectordb-rag/outputs/retrieval_eval.csv (75행)


### 📝 Step 12 관찰 메모
- 최종 순위: **Hybrid+Rerank(0.867) > BM25 한국어혼합(0.800) ≈ Hybrid(0.787) > Dense(0.693) > BM25 공백(0.653)**
- `search()`는 Dense(Qdrant 필터)와 BM25(`weight_mask`)에 **같은 조건**을 걸어서 필터를 적용한다. 한쪽에만 필터를 걸면 RRF에서 조건 밖의 청크가 섞여 들어오기 때문이다.
- `year >= 2026` 필터 결과도 1등은 "블랙웰"이 아니라 "H200" 청크였다. 필터는 "범위"를 좁혀줄 뿐 순위 품질은 여전히 검색·Rerank가 결정한다.
- 남은 과제: ① 키워드 근사 지표를 **실제 답변 품질 지표(RAGAS)**로 교체 ② 영문/한글 표기 차이(DeepSeek ↔ 딥시크) 대응 ③ Rerank 속도. ①은 04에서 다룬다.

## ✅ Step 7~12 체크포인트

- [x] Step 8: BM25 인덱스 2개(공백 / 한국어혼합), 토크나이저 차이 확인
- [x] Step 9: 평가 세트 15문항으로 Dense 실패 사례(MLPerf 등) 확인
- [x] Step 10: RRF Hybrid 성능 측정
- [x] Step 11: Reranker 재정렬, top_n별 정확도·속도 비교
- [x] Step 12: 최종 비교표, 필터 지원 `search()`, `outputs/eval_queries.json`·`retrieval_eval.csv` 저장

### 다음: `04_ragas_eval.ipynb`
`search()`로 찾은 청크를 LLM에 넣어 답변 생성 → RAGAS(Faithfulness, Context Precision 등)로 Dense vs Hybrid+Rerank의 **답변 품질** Before/After 비교